# 03 — Model Comparison & Error Analysis (TV6 Evaluation)

Notebook phục vụ phân tích so sánh các mô hình và khảo sát sai số chi tiết theo đúng quy định tại `AGENTS.md`:
- **Quy tắc 1:** Chỉ so sánh các mô hình trên tập `validation` (tuyệt đối không dùng tập `test` để chọn model).
- **Quy tắc 2:** Các mô hình phải được đánh giá trên cùng một tập dữ liệu (`population_id` đồng nhất).
- **Quy tắc 3:** Mọi số liệu và biểu đồ đo lường bằng đơn vị vật lý **°C**.

## 1. So sánh tổng quan các mô hình (Validation Metrics)

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# Điền đường dẫn các file metrics.json từ các validation run thật
# Ví dụ: ["runs/lstm/evaluation/metrics.json", "runs/attention/evaluation/metrics.json"]
metric_paths = []

if not metric_paths:
    print("Chưa có runs thực tế. Vui lòng cung cấp danh sách metric_paths khi có kết quả huấn luyện.")
else:
    payloads = [json.loads(Path(p).read_text(encoding="utf-8")) for p in metric_paths]
    if any(item["split"] != "validation" for item in payloads):
        raise ValueError("Model comparison chỉ được sử dụng tập validation!")
    if len({item["population_id"] for item in payloads}) != 1:
        raise ValueError("Các mô hình không được đánh giá trên cùng population_id!")

    comparison_df = pd.DataFrame([{
        "run_id": item["run_id"],
        "model_name": item["model_name"],
        "mae_deg_c": item["overall"]["mae"],
        "rmse_deg_c": item["overall"]["rmse"],
        "directional_acc": item["overall"].get("directional_accuracy", None),
    } for item in payloads]).sort_values("rmse_deg_c")
    display(comparison_df)

## 2. Phân tích sai số theo Horizon (1h - 72h)

In [ ]:
# Đọc file metrics_per_horizon.csv từ thư mục evaluation
# horizon_csv = Path("runs/<run_id>/evaluation/metrics_per_horizon.csv")
horizon_csv = None

if horizon_csv and horizon_csv.exists():
    horizon_df = pd.read_csv(horizon_csv)
    plt.figure(figsize=(10, 4))
    plt.plot(horizon_df["horizon"], horizon_df["rmse"], label="RMSE (°C)", color="crimson")
    plt.plot(horizon_df["horizon"], horizon_df["mae"], label="MAE (°C)", color="navy")
    plt.xlabel("Forecast Horizon (hours ahead)")
    plt.ylabel("Error (°C)")
    plt.title("Error Breakdown across 72h Forecast Horizon")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("Chưa có metrics_per_horizon.csv để vẽ biểu đồ.")

## 3. Phân tích sai số theo Chu kỳ ngày & Mùa (Diurnal & Seasonal)

In [ ]:
# error_by_hour_csv = Path("runs/<run_id>/evaluation/error_by_hour.csv")
# error_by_season_csv = Path("runs/<run_id>/evaluation/error_by_season.csv")
print("Phân tích sai số theo giờ và mùa sẽ được nạp từ file kết quả error_analysis.")

## 4. Khảo sát các trường hợp sai số lớn nhất (Worst-Case Analysis)

In [ ]:
# worst_cases_csv = Path("runs/<run_id>/evaluation/worst_cases.csv")
print("Bảng top-20 trường hợp sai số lớn nhất giúp chẩn đoán các sự kiện thời tiết cực đoan.")